# Wave 01 floor on Kaggle

Runs `FLOOR_PROTOCOL.md` for all seven base models on the GPU: same data, same prompt format, same decoding and scoring as the laptop runs, recorded with `machine=kaggle` and fp16.

Settings this notebook needs (right-hand panel): **Accelerator: GPU T4 x2** (or P100), **Internet: On**. Then *Run All*. When it finishes, download `/kaggle/working/eliza_floor_runs.zip` from the Output panel.

In [ ]:
REPO = 'https://github.com/gozonerd/port-mini-models-eliza-post-training'
COMMIT = 'master'   # the repo's branch; pin to a commit hash to reproduce exactly (the run prints the hash it used)
import subprocess, os
subprocess.run(['git', 'clone', '--quiet', REPO, '/kaggle/working/repo'], check=True)
os.chdir('/kaggle/working/repo')
subprocess.run(['git', 'checkout', '--quiet', COMMIT], check=True)
print('repo at', subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip())
subprocess.run(['pip', 'install', '--quiet', 'transformers>=4.45', 'accelerate', 'peft'], check=True)
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# the seven pinned base checkpoints, smallest first (FLOOR_PROTOCOL.md)
MODELS = [
    ('EleutherAI/pythia-70m',      'a39f36b100fe8a5377810d56c3f4789b9c53ac42'),
    ('HuggingFaceTB/SmolLM2-135M', '93efa2f097d58c2a74874c7e644dbc9b0cee75a2'),
    ('EleutherAI/pythia-160m',     '50f5173d932e8e61f858120bcb800b97af589f46'),
    ('HuggingFaceTB/SmolLM2-360M', 'f8027fd0eaeea54caa13c31d31b9fdc459c38b49'),
    ('EleutherAI/pythia-410m',     '9879c9b5f8bea9051dcb0e68dff21493d67e9d4f'),
    ('EleutherAI/pythia-1b',       'f73d7dcc545c8bd326d8559c8ef84ffe92fea6b2'),
    ('HuggingFaceTB/SmolLM2-1.7B', 'effd688a12921b4cc83e3312b6feb579f70f9c71'),
]
import subprocess, sys, time
for model, rev in MODELS:
    t0 = time.time()
    print('=====', model, rev, flush=True)
    r = subprocess.run([sys.executable, 'tools/measure_floor.py', '--model', model, '--revision', rev,
                        '--split', 'test', '--device', 'cuda', '--batch', '32', '--out', 'runs',
                        '--note', 'floor, FLOOR_PROTOCOL.md, kaggle GPU fp16'], capture_output=True, text=True)
    print(r.stdout[-1500:]); print(r.stderr[-800:] if r.returncode else '', flush=True)
    print(f'{model}: {time.time()-t0:.0f}s', flush=True)

In [ ]:
import subprocess
print(open('runs/eliza_runs.tsv').read())
subprocess.run(['zip', '-qr', '/kaggle/working/eliza_floor_runs.zip', 'runs'], check=True)
print('download /kaggle/working/eliza_floor_runs.zip from the Output panel')